In [1]:
import logging
import sys

# Konfiguration für Jupyter erzwingen
logging.basicConfig(
    level=logging.DEBUG,  # Auf welchen Level wird geloggt 
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout, 
    force=True         
)

# Logger erstellen
logger = logging.getLogger(__name__)

In [ ]:
# ============================================================
# HAZMAT VRP SOLVER 
# ============================================================
import json
import numpy as np
import pandas as pd
import pulp as pl
import time as time_module


# ------------------------------------------------------------
# 0. GLOBALE GEWICHTE 
#    w1 = Risiko, w2 = Kosten, w3 = Zeit 
# ------------------------------------------------------------
t0 = time_module.perf_counter() 

W_RISK, W_COST, W_TIME = 0.5, 0.3, 0.2


# ============================================================
# 1. DATEN LADEN
# ============================================================
# Hier müssen die 4 verschiedenen Dateien verwendet werden. 1. Routen Matrix 2. Charger Matrix 3. Vehicles 4. Kunden alle zu finden 
# in Sciebo, jeweils in den Unterordnern "small", "medium", "large" 
hazmat_df = pd.read_csv(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Data\OperationsResearch\od_matrix_small.csv")
charger_df = pd.read_csv(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Data\OperationsResearch\od_matrix_small_charger.csv")
df_vehicles = pd.read_csv(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\data\processed\vehicles.csv")
df_kunden = pd.read_csv(r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\Data\OperationsResearch\small_instanz.csv")

df_kunden_raw = df_kunden.copy()   # Rohdaten sicher, für späteren Vergleich 


DEPOT = "DEPOT"
df_kunden = df_kunden[df_kunden["id"] != DEPOT].copy()
df_kunden["quantity"] = pd.to_numeric(df_kunden["quantity"])

customers = df_kunden["id"].tolist()
Demand = dict(zip(df_kunden["id"], df_kunden["quantity"]))

vehicles = df_vehicles["type"].tolist()
Cap = dict(zip(df_vehicles["type"], df_vehicles["fuel_capacity_l"]))
MaxRange = dict(zip(df_vehicles["type"], df_vehicles["range_km"]))
FixCost = dict(zip(df_vehicles["type"], df_vehicles["fixcost"]))
VarCost = dict(zip(df_vehicles["type"], df_vehicles["variable_cost_per_km"]))

#------------------------------------
# HEURISTIK LADEN (WARM START)
# -----------------------------------
heuristic_file = r"C:\Users\j.beckmann\OneDrive - Reply\Uni Jonas Beckmann\4.Semester\Operations Research\Projekt\Umwelftfreundliche_Routenplanung\models\heuristic_small_risk_oriented_9_customers.json"

try:
    with open(heuristic_file, 'r', encoding='utf-8') as f:
        heuristic_data = json.load(f)
    heuristic_routes = heuristic_data.get("routes", {})
    logger.debug(f"Heuristik erfolgreich geladen. {len(heuristic_routes)} Fahrzeug-Routen gefunden.")
except Exception as e:
    logger.error(f"Fehler beim Laden der Heuristik: {e}")
    heuristic_routes = {}

t1 = time_module.perf_counter() # Ende Datenvorbereitung
# ============================================================
# 2. KANTENAUFBEREITUNG 
# ============================================================


loaded = hazmat_df[
    (hazmat_df["profile"] == "safest") & (hazmat_df["load_state"] == "loaded")
].copy()
# laden aller Straßenprofile (safest, fastest, shortest) 
loaded = hazmat_df[hazmat_df["load_state"] == "loaded"].copy()

# Relationen, deren einziger Pfad einen gesperrten Tunnel nutzt, haben cost = inf 
# Falls vorhanden 
prem = loaded["cost"] - loaded["dist_km"]
prem = np.maximum(0, prem)  # nur positive Risikoprämien
finite_prem = prem[np.isfinite(prem)]
RISK_PENALTY = 2.0 * finite_prem.max() if finite_prem.max() > 0 else 9999.0
n_adr_violations = int((~np.isfinite(prem)).sum())
loaded["risk_prem"] = prem.where(np.isfinite(prem), RISK_PENALTY)

logger.debug(f"ADR-kritische Kanten (cost=inf, nur Tunnelroute): {n_adr_violations}"
      f" -> Risikoprämie {RISK_PENALTY:.1f} km-Äquivalente")


# Maximalwerte für die Normierung  
max_r_l = loaded["risk_prem"].max() if loaded["risk_prem"].max() > 0 else 1.0
max_d_l = loaded["dist_km"].max()
max_t_l = loaded["time_min"].max()

# eval_cost für jede mögliche Strecke
loaded["eval_cost"] = (
    W_RISK * loaded["risk_prem"] / max_r_l
    + W_COST * loaded["dist_km"] / max_d_l
    + W_TIME * loaded["time_min"] / max_t_l
)

#Auswahl bestes Profil (min eval_cost)
loaded_best = loaded.loc[loaded.groupby(["from", "to"])["eval_cost"].idxmin()].copy()

# Leerfahrten: bestes Profil je Relation (nur Distanz/Zeit, Risiko = 0)
empty = hazmat_df[hazmat_df["load_state"] == "empty"].copy()
max_d_e = empty["dist_km"].max()
max_t_e = empty["time_min"].max()
empty["eval_cost"] = (
    W_COST * empty["dist_km"] / max_d_e + W_TIME * empty["time_min"] / max_t_e
)
# Auswahl bestes Profil (min eval_cost)
empty_best = empty.loc[empty.groupby(["from", "to"])["eval_cost"].idxmin()].copy()

# ------------------------------------------------------------
# Routing-Kanten-Dictionaries
# ------------------------------------------------------------
dist, time, risk = {}, {}, {}

def add_arc(i, j, d, t, r):
    dist[(i, j)] = round(float(d), 4)
    time[(i, j)] = round(float(t), 4)
    risk[(i, j)] = round(float(r), 4)

# Beladene Kanten: Depot->Kunde und Kunde->Kunde 
for _, row in loaded_best.iterrows():
    i, j = row["from"], row["to"]
    if j == DEPOT:
        continue  # Rückfahrt wird als Leerfahrt modelliert
    add_arc(i, j, row["dist_km"], row["time_min"], row["risk_prem"])

# Leerkanten: Kunde -> Depot (risikofrei, frei geroutet)
for _, row in empty_best.iterrows():
    i, j = row["from"], row["to"]
    if j != DEPOT or i == DEPOT:
        continue
    add_arc(i, DEPOT, row["dist_km"], row["time_min"], 0.0)

# ------------------------------------------------------------
# Ladesäulen-Kanten (Side-Trips), auch nach besten Profil
# ------------------------------------------------------------
chg = charger_df.copy()
max_d_c = chg["dist_km"].max()
max_t_c = chg["time_min"].max()
max_r_c = chg["risk"].max() if chg["risk"].max() > 0 else 1.0

# eval_cost für jede mögliche Strecke
chg["eval_cost"] = (
    W_RISK * chg["risk"] / max_r_c
    + W_COST * chg["dist_km"] / max_d_c
    + W_TIME * chg["time_min"] / max_t_c
)
# Bestes Profil 
chg_best = chg.loc[chg.groupby(["from", "to"])["eval_cost"].idxmin()].copy()

# Dictionaries erstellen 
c_dist, c_time, c_risk = {}, {}, {}
for _, row in chg_best.iterrows():
    i, j = row["from"], row["to"]
    c_dist[(i, j)] = round(float(row["dist_km"]), 4)
    c_time[(i, j)] = round(float(row["time_min"]), 4)
    c_risk[(i, j)] = round(float(row["risk"]), 4)

# Side-Trip nur möglich, wenn Hin- und Rückkante real existieren
side_pairs = sorted(
    {(i, j) for (i, j) in c_dist if i in customers and (j, i) in c_dist}
)
stations = sorted({s for (_, s) in side_pairs})
cust_no_charger = [c for c in customers if not any(cc == c for (cc, _) in side_pairs)]
logger.debug(f"Side-Trip-Paare (Kunde<->Säule, beide Richtungen): {len(side_pairs)}")
logger.debug(f"Kunden ohne Ladesäulen-Anbindung: {cust_no_charger}")

# Gesamt-Kantenübersicht 
od_merged = pd.concat([loaded_best, empty_best, chg_best], ignore_index=True)

# ============================================================
# 3. SETS & PARAMETER
# ============================================================

routing_nodes = customers + [DEPOT]
routing_arcs = list(dist.keys())
logger.debug(f"Routing-Knoten: {len(routing_nodes)} | Routing-Kanten: {len(routing_arcs)}")

# ---- Lenk-, Ruhe- und Schichtzeiten ----
MAX_DRIVING_BLOCK = 270.0   # 4,5 h maximale Lenkzeit am Stück (Art. 7)
MAX_DAILY_DRIVING = 540.0   # 9 h maximale Tageslenkzeit (Art. 6)
MAX_SHIFT_TIME = 780.0      # 13 h Schichtfenster (24 h - 11 h Ruhezeit, Art. 8)
BREAK_TIME = 45.0           # 45 min Fahrtunterbrechung (Art. 7)

SERVICE_TIME = 45.0         # 0,75 h Servicezeit je Kunde 
RECHARGE_TIME = 45.0        # HPC-Ladevorgang (zählt als Pause)
CHARGE_COST = 300.0         # Pauschale Ladekosten je Ladevorgang

ServiceTime = {c: SERVICE_TIME for c in customers}
ServiceTime[DEPOT] = 0.0

# ---- Big-M je Constraint-Typ ----
max_arc_time = max(time.values())
max_side_time = max(
    (c_time[(c, s)] + c_time[(s, c)] + RECHARGE_TIME for (c, s) in side_pairs),
    default=0.0,
)
BIG_M_T = MAX_SHIFT_TIME + SERVICE_TIME + max_side_time + BREAK_TIME + max_arc_time
BIG_M_D = MAX_DRIVING_BLOCK + max_arc_time
BIG_M_B = max(MaxRange.values())
BIG_M_LOAD = max(Cap.values())

t2 = time_module.perf_counter()  # Ende Kantenaufbereitung
# ============================================================
# 4. MODELL & VARIABLEN
# ============================================================

model = pl.LpProblem("Hazmat_VRP_SingleTrip", pl.LpMinimize)

#Kanten aktivieren: 1 => Fahrzeug v nutzt Kante, 0 => nutzt nicht 
x = pl.LpVariable.dicts(
    "x", [(i, j, v) for (i, j) in routing_arcs for v in vehicles], cat="Binary"
)

# Ankunftszeit an Knoten n (min seit Schichtbeginn)
T = pl.LpVariable.dicts(
    "T", [(n, v) for n in routing_nodes for v in vehicles],
    lowBound=0, upBound=MAX_SHIFT_TIME,
)

# Batteriezustand bei Ankunft an Knoten n (km Reichweite) 
B_arr = pl.LpVariable.dicts(
    "B_arr", [(n, v) for n in routing_nodes for v in vehicles],
    lowBound=0, upBound=BIG_M_B,
)
# Batteriezustand bei Abfahrt von Knoten n (falls geladen wird)  
B_dep = pl.LpVariable.dicts(
    "B_dep", [(n, v) for n in routing_nodes for v in vehicles],
    lowBound=0, upBound=BIG_M_B,
)

# Restladung nach Besuch Knoten n (Liter)
Load = pl.LpVariable.dicts(
    "Load", [(n, v) for n in routing_nodes for v in vehicles], lowBound=0
)

# Side-Trip: Fahrzeug v lädt an Säule s vom Kunden c 
y = pl.LpVariable.dicts(
    "y", [(c, s, v) for (c, s) in side_pairs for v in vehicles], cat="Binary"
)

# Reine Lenkpause (45 min) am Kunden c
p = pl.LpVariable.dicts(
    "p", [(c, v) for c in customers for v in vehicles], cat="Binary"
)

# Lenkzeit seit letzter Pause (Ankunft / Abfahrt)
D_arr = pl.LpVariable.dicts(
    "D_arr", [(n, v) for n in routing_nodes for v in vehicles],
    lowBound=0, upBound=MAX_DRIVING_BLOCK,
)
D_dep = pl.LpVariable.dicts(
    "D_dep", [(n, v) for n in routing_nodes for v in vehicles],
    lowBound=0, upBound=MAX_DRIVING_BLOCK,
)

# Hilfsausdrücke
z = {  # Fahrzeug v verlässt das Depot (= Fahrzeug wird eingesetzt)
    v: pl.lpSum(x[(DEPOT, j, v)] for j in customers if (DEPOT, j) in dist)
    for v in vehicles
}
visit = {  # Fahrzeug v besucht Kunde c
    (c, v): pl.lpSum(x[(i, c, v)] for i in routing_nodes if (i, c) in dist)
    for c in customers for v in vehicles
}

# Bündelt Gesamtzeit, Fahrzeit, Distanz und Risiko aller Side-Trips am Kunden c
# Kann dann bei den Constraints einfach aufgerufen werden 
def side_terms(c, v):
    pairs = [(cc, s) for (cc, s) in side_pairs if cc == c]
    t_expr = pl.lpSum(
        (c_time[(c, s)] + c_time[(s, c)] + RECHARGE_TIME) * y[(c, s, v)]
        for (_, s) in pairs
    )
    drive_expr = pl.lpSum(
        (c_time[(c, s)] + c_time[(s, c)]) * y[(c, s, v)] for (_, s) in pairs
    )
    d_expr = pl.lpSum(
        (c_dist[(c, s)] + c_dist[(s, c)]) * y[(c, s, v)] for (_, s) in pairs
    )
    r_expr = pl.lpSum(
        (c_risk[(c, s)] + c_risk[(s, c)]) * y[(c, s, v)] for (_, s) in pairs
    )
    return t_expr, drive_expr, d_expr, r_expr

# ============================================================
# 5. ZIELFUNKTION: EIN normierter Ausdruck
#    min  w1*Risk/Rmax + w2*Cost/Cmax + w3*Time/Tmax
# ============================================================
# Summieren des Risiko der Strecken 
risk_expr = pl.lpSum(
    risk[(i, j)] * x[(i, j, v)] for (i, j) in routing_arcs for v in vehicles
) + pl.lpSum(
    (c_risk[(c, s)] + c_risk[(s, c)]) * y[(c, s, v)]
    for (c, s) in side_pairs for v in vehicles
)

# Summieren der Kosten: Fixkosten + variable Kosten je Strecke + Side-Trip-Kosten
travel_cost = pl.lpSum(
    VarCost[v] * dist[(i, j)] * x[(i, j, v)]
    for (i, j) in routing_arcs for v in vehicles
)
fixed_cost = pl.lpSum(FixCost[v] * z[v] for v in vehicles)
side_trip_cost = pl.lpSum(
    (VarCost[v] * (c_dist[(c, s)] + c_dist[(s, c)]) + CHARGE_COST) * y[(c, s, v)]
    for (c, s) in side_pairs for v in vehicles
)
cost_expr = travel_cost + fixed_cost + side_trip_cost

# Gesamtdauer: Fahrzeit + Servicezeit (Kunden) + Lade-Abstecher + Pausen
time_expr = (
    pl.lpSum(time[(i, j)] * x[(i, j, v)] for (i, j) in routing_arcs for v in vehicles)
    + pl.lpSum(ServiceTime[j] * x[(i, j, v)]
               for (i, j) in routing_arcs if j != DEPOT for v in vehicles)
    + pl.lpSum((c_time[(c, s)] + c_time[(s, c)] + RECHARGE_TIME) * y[(c, s, v)]
               for (c, s) in side_pairs for v in vehicles)
    + pl.lpSum(BREAK_TIME * p[(c, v)] for c in customers for v in vehicles)
)

# Normierung der Zielwerte auf [0,1] (maximaler Wert = Worst-Case)
max_sol_arcs = len(customers) + len(vehicles)
RISK_SCALE = max(risk.values()) * max_sol_arcs
if RISK_SCALE <= 0:
    RISK_SCALE = 1.0
COST_SCALE = (
    sum(FixCost.values())
    + max(dist.values()) * max(VarCost.values()) * max_sol_arcs
    + CHARGE_COST * len(customers)
)
if COST_SCALE <= 0:
    COST_SCALE = 1.0
    
TIME_SCALE = MAX_SHIFT_TIME * len(vehicles)
if TIME_SCALE <= 0:
    TIME_SCALE = 1.0

# zusammensetzen der Zielfunktion 
model += (
    W_RISK * risk_expr / RISK_SCALE
    + W_COST * cost_expr / COST_SCALE
    + W_TIME * time_expr / TIME_SCALE
)

# ============================================================
# 6. ROUTING-CONSTRAINTS
# ============================================================

# C1: Jeder Kunde genau einmal
for c in customers:
    model += pl.lpSum(visit[(c, v)] for v in vehicles) == 1

# C5: Flusserhaltung an Kunden
for v in vehicles:
    for n in customers:
        model += (
            pl.lpSum(x[(i, n, v)] for i in routing_nodes if (i, n) in dist)
            == pl.lpSum(x[(n, j, v)] for j in routing_nodes if (n, j) in dist)
        )

# C7: Depot als Start/Ende, max. 1 Abfahrt je Fahrzeug (=genau 1 Tour, kein Reload)
for v in vehicles:
    model += z[v] == pl.lpSum(
        x[(i, DEPOT, v)] for i in customers if (i, DEPOT) in dist
    )
    model += z[v] <= 1

# Gültiger Schnitt: eingesetzte Kapazität >= Gesamtnachfrage
model += pl.lpSum(Cap[v] * z[v] for v in vehicles) >= sum(Demand.values())

# C3: Kapazität über Load-Propagation (Reset am Depot)
for v in vehicles:
    model += Load[(DEPOT, v)] == Cap[v]
    for (i, j) in routing_arcs:
        if j in Demand:
            model += (
                Load[(j, v)] <= Load[(i, v)] - Demand[j]
                + BIG_M_LOAD * (1 - x[(i, j, v)])
            )

# Entweder reine Pause ODER Lade-Side-Trip, und nur bei Besuch
for v in vehicles:
    for c in customers:
        model += (
            p[(c, v)]
            + pl.lpSum(y[(c, s, v)] for (cc, s) in side_pairs if cc == c)
            <= visit[(c, v)]
        )

# ============================================================
# 6.5 SYMMETRY BREAKING (identische Fahrzeuge)
# ============================================================
# Identische Fahrzeuge werden in Gruppen zusammengefasst, und die Variablen z[v] werden in der Reihenfolge der Gruppen eingefroren
#  Zwingt den Solver, Fahrzeuge einer Gruppe strikt nacheinander einzusetzen 
vehicle_groups = df_vehicles.groupby(
    ["capacity_t", "battery_kwh", "range_km", "fixcost", "variable_cost_per_km"]
)["type"].apply(list).tolist()

for group in vehicle_groups:
    if len(group) > 1:
        logger.debug(f"Identische Fahrzeuggruppe: {group}")
        for v1, v2 in zip(group, group[1:]):
            model += z[v1] >= z[v2]

# ============================================================
# 7. ZEIT — Subtour-Eliminierung über Zeitpropagation
# ============================================================

for v in vehicles:
    #Schichtstart bei Depot = 0 min
    model += T[(DEPOT, v)] == 0

    for (i, j) in routing_arcs:
        if j == DEPOT:
            continue
        if i in customers:
            side_t, _, _, _ = side_terms(i, v)
            extra = side_t + BREAK_TIME * p[(i, v)]
        else:
            extra = 0
        # Subtour-Eliminierung: Ankunftszeit an j >= Ankunftszeit an i + ServiceTime + Fahrzeit + SideTrip + Pause
        model += (
            T[(j, v)] >= T[(i, v)] + ServiceTime[i] + extra + time[(i, j)]
            - BIG_M_T * (1 - x[(i, j, v)])
        )

    # Schichtende: Rückkehr am Depot <= 13 h
    for i in customers:
        if (i, DEPOT) in dist:
            side_t, _, _, _ = side_terms(i, v)
            model += (
                T[(i, v)] + ServiceTime[i] + side_t + BREAK_TIME * p[(i, v)]
                + time[(i, DEPOT)]
                <= MAX_SHIFT_TIME + BIG_M_T * (1 - x[(i, DEPOT, v)])
            )

# ============================================================
# 8. EU-LENKZEITEN 
# ============================================================

for v in vehicles:
    # Lenkzeit-Stoppuhr startet am Depot bei 0
    model += D_dep[(DEPOT, v)] == 0

    # Lenkzeit-Stoppuhr startet bei Ankunft an Knoten n = Lenkzeit seit letzter Pause
    for (i, j) in routing_arcs:
        model += (
            D_arr[(j, v)] >= D_dep[(i, v)] + time[(i, j)]
            - BIG_M_D * (1 - x[(i, j, v)])
        )


    for c in customers:
        pairs_c = [(cc, s) for (cc, s) in side_pairs if cc == c]
        is_charging = pl.lpSum(y[(c, s, v)] for (_, s) in pairs_c)

        # Ohne Pause/Laden: Lenkzeit wird durchgereicht
        model += D_dep[(c, v)] <= D_arr[(c, v)] + BIG_M_D * (p[(c, v)] + is_charging)
        model += D_dep[(c, v)] >= D_arr[(c, v)] - BIG_M_D * (p[(c, v)] + is_charging)

        # Reine Pause (p=1): Reset auf 0
        model += D_dep[(c, v)] <= BIG_M_D * (1 - p[(c, v)])

        # Lade-Side-Trip (y=1): Anfahrt zur Säule muss legal sein,
        # nach dem Laden zählt nur die Rückfahrt von der Säule
        for (_, s) in pairs_c:
            model += (
                D_arr[(c, v)] + c_time[(c, s)]
                <= MAX_DRIVING_BLOCK + BIG_M_D * (1 - y[(c, s, v)])
            )
            model += (
                D_dep[(c, v)] >= c_time[(s, c)] - BIG_M_D * (1 - y[(c, s, v)])
            )
            model += (
                D_dep[(c, v)] <= c_time[(s, c)] + BIG_M_D * (1 - y[(c, s, v)])
            )

    # Tageslenkzeit max. 9 h (inkl. Side-Trips)
    driving_main = pl.lpSum(time[(i, j)] * x[(i, j, v)] for (i, j) in routing_arcs)
    driving_side = pl.lpSum(
        (c_time[(c, s)] + c_time[(s, c)]) * y[(c, s, v)]
        for (c, s) in side_pairs
    )
    model += driving_main + driving_side <= MAX_DAILY_DRIVING

# ============================================================
# 9. BATTERIE & REICHWEITE 
# ============================================================

for v in vehicles:
    # Abfahrt Depot mit voller Batterie (Laden während Beladung)
    model += B_dep[(DEPOT, v)] == MaxRange[v]

    for n in routing_nodes:
        model += B_arr[(n, v)] <= MaxRange[v]
        model += B_dep[(n, v)] <= MaxRange[v]

    # Verbrauch auf allen Kanten (nur reale Distanzen)
    for (i, j) in routing_arcs:
        model += (
            B_arr[(j, v)] <= B_dep[(i, v)] - dist[(i, j)]
            + BIG_M_B * (1 - x[(i, j, v)])
        )

    for c in customers:
        pairs_c = [(cc, s) for (cc, s) in side_pairs if cc == c]
        is_charging = pl.lpSum(y[(c, s, v)] for (_, s) in pairs_c)

        # Batterie muss für Hinfahrt zur Säule reichen
        model += B_arr[(c, v)] >= pl.lpSum(
            c_dist[(c, s)] * y[(c, s, v)] for (_, s) in pairs_c
        )

        # Ohne Laden: Abfahrtsbatterie <= Ankunftsbatterie
        model += B_dep[(c, v)] <= B_arr[(c, v)] + BIG_M_B * is_charging

        # Mit Laden an s: Abfahrt <= MaxRange - Rückweg von der Säule
        for (_, s) in pairs_c:
            model += (
                B_dep[(c, v)] <= MaxRange[v] - c_dist[(s, c)]
                + BIG_M_B * (1 - y[(c, s, v)])
            )

# ============================================================
# 10. MODELLGRÖSSE & DIAGNOSE
# ============================================================

logger.debug("----------------------")
logger.debug("Modellgröße")
logger.debug(f"Anzahl x-Variablen: {len(x)}")
logger.debug(f"Anzahl y-Variablen: {len(y)}")
logger.debug(f"Anzahl p-Variablen: {len(p)}")
logger.debug(f"Gesamtzahl Variablen: {len(model.variables())}")
logger.debug(f"Anzahl Constraints: {len(model.constraints)}")
logger.debug(f"Fahrzeuge: {len(vehicles)} | Kunden: {len(customers)} | Stationen (real angebunden): {len(stations)}")
logger.debug("---------------------")

logger.debug("Plausibilität der Daten")
logger.debug(f"Gesamtnachfrage [l]: {sum(Demand.values())}")
logger.debug(f"Gesamtkapazität je Fahrzeug [l]: {sum(Cap.values())}")
logger.debug(f"NaN im Demand?: {any(pd.isna(v) for v in Demand.values())}")
logger.debug(f"NaN in dist/time/risk?: {any(pd.isna(v) for v in dist.values())}, {any(pd.isna(v) for v in time.values())}, {any(pd.isna(v) for v in risk.values())}")
logger.debug("----------------------")



# ============================================================
# 11. SOLVER
# ============================================================
t3 = time_module.perf_counter() # Ende Modellaufbau

# --- WARM START LOGIK ---
if heuristic_routes:
    print("Wende Warm Start aus Heuristik an...")
    
    # Alle x-Variablen auf 0 setzen
    for var in x.values():
        var.setInitialValue(0)
        
    # Die Kanten aus der Heuristik auf 1 setzen
    for v, route in heuristic_routes.items():
        if v not in vehicles:
            continue  
            
        # Durchlaufen der Route und Setzen der Kanten
        for idx in range(len(route) - 1):
            i = route[idx]
            j = route[idx + 1]
            
            # Prüfen, ob diese Kante im Modell existiert
            if (i, j, v) in x:
                x[(i, j, v)].setInitialValue(1)
            else:

                logger.warning(f"Warnung: Heuristik-Kante {i} -> {j} für {v} existiert nicht in x-Variablen!")
# -------------------------


print("Starte Optimierung...")
solver = pl.PULP_CBC_CMD(
    msg=1,
    timeLimit=600, 
    gapRel=0.02,
    logPath="cbc_log.txt",
    warmStart=True
)
model.solve(solver)

t4 = time_module.perf_counter() # Ende Solver Laufzeit

print("\nStatus:", pl.LpStatus[model.status])
print("Objective (normiert):", pl.value(model.objective))

# Ausgabe der CBC-Logdatei, um zu wissen, ob die Lösung optimal oder nur zulässig ist
try:
    with open("cbc_log.txt") as f:
        for line in f:
            if line.startswith(("Result", "Objective value", "Lower bound", "Gap:")):
                print("CBC:", line.strip())
except OSError:
    pass

# ============================================================
# 12. ROUTEN & KENNZAHLEN
# ============================================================

def build_route(v):
    """Folgt den aktiven x-Kanten ab Depot."""
    route = [DEPOT]
    current = DEPOT
    steps = 0

    # Maximale Anzahl Schritte = 2 * Anzahl Knoten, um Endlosschleifen zu vermeiden
    while steps <= 2 * len(routing_nodes):
        steps += 1
        nxt = [
            j for j in routing_nodes
            if (current, j) in dist and pl.value(x[(current, j, v)]) is not None
            and pl.value(x[(current, j, v)]) > 0.5
        ]
        if not nxt:
            break
        route.append(nxt[0])
        if nxt[0] == DEPOT:
            break
        current = nxt[0]
    return route

def build_route_with_charging(v):
    base = build_route(v)
    full = []
    for node in base:
        full.append(node)
        if node in customers:
            for (c, s) in side_pairs:
                if c == node and pl.value(y[(c, s, v)]) is not None \
                        and pl.value(y[(c, s, v)]) > 0.5:
                    full.extend([s, node])
        if node in customers and pl.value(p[(node, v)]) is not None \
                and pl.value(p[(node, v)]) > 0.5:
            full.append("PAUSE")
    return full

if model.status == 1:
    print("\nROUTEN:")
    for v in vehicles:
        r = build_route_with_charging(v)
        if len(r) > 2:
            served = [c for c in customers if pl.value(visit[(c, v)]) > 0.5]
            load_l = sum(Demand[c] for c in served)
            drive = pl.value(
                pl.lpSum(time[(i, j)] * x[(i, j, v)] for (i, j) in routing_arcs)
            )
            service_total = pl.value(
                pl.lpSum(ServiceTime[j] * x[(i, j, v)]
                         for (i, j) in routing_arcs if j != DEPOT)
            )
            charge_total = pl.value(
                pl.lpSum((c_time[(c, s)] + c_time[(s, c)] + RECHARGE_TIME) * y[(c, s, v)]
                         for (c, s) in side_pairs)
            )
            break_total = pl.value(
                pl.lpSum(BREAK_TIME * p[(c, v)] for c in customers)
            )
            vehicle_total_time = drive + service_total + charge_total + break_total

            print(f"{v}: {' -> '.join(r)}")
            print(f"\tKunden: {len(served)} | Menge: {load_l} l | "
                  f"reine Fahrzeit: {drive:.0f} min | "
                  f"Arbeitszeit: {vehicle_total_time:.0f} min "
                  f"(Service: {service_total:.0f}, Laden: {charge_total:.0f}, "
                  f"Pause: {break_total:.0f})")
        else:
            print(f"{v}: nicht genutzt")

    print("\nZeit/Lenkzeit/Batterie je Fahrzeug :")
    for v in vehicles:
        r = build_route(v)
        if len(r) <= 2:
            continue
        print(f"  {v}:")
        for n in r[1:]:
            if n == DEPOT:
                continue
            print(f"\t{n}: Zeit bei Ankunft={pl.value(T[(n, v)]):.0f} min | "
                  f"Verstrichene Zeit bis Pause={pl.value(D_arr[(n, v)]):.0f}/{MAX_DRIVING_BLOCK:.0f} min | "
                  f"Reichweite={pl.value(B_arr[(n, v)]):.0f} km | "
                  f"Restkapazität={pl.value(Load[(n, v)]):.0f} l")

    print("\nZielfunkstionswerte (unnormiert):")
    print(f"  Risiko gesamt [km-Äquiv.]: {pl.value(risk_expr):.1f}")
    print(f"  Kosten gesamt [EUR]:       {pl.value(cost_expr):.2f}")
    print(f"\tdavon Fixkosten:         {pl.value(fixed_cost):.2f}")
    print(f"\tdavon variable Kosten:   {pl.value(travel_cost):.2f}")
    print(f"\tdavon Laden (Side-Trips):{pl.value(side_trip_cost):.2f}")
    print(f"  Einsatzzeit gesamt [min]:  {pl.value(time_expr):.1f}")
else:
    print("\nKeine gültige Lösung — Status:", pl.LpStatus[model.status])

t5 = time_module.perf_counter() # Ende Ausgabe 

18:23:26 - numexpr.utils - INFO - Note: NumExpr detected 16 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
18:23:26 - numexpr.utils - INFO - NumExpr defaulting to 8 threads.


c:\Users\j.beckmann\AppData\Local\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\j.beckmann\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


18:23:29 - __main__ - DEBUG - Heuristik erfolgreich geladen. 2 Fahrzeug-Routen gefunden.
18:23:29 - __main__ - DEBUG - ADR-kritische Kanten (cost=inf, nur Tunnelroute): 60 -> Risikoprämie 108.8 km-Äquivalente
18:23:29 - __main__ - DEBUG - Side-Trip-Paare (Kunde<->Säule, beide Richtungen): 29
18:23:29 - __main__ - DEBUG - Kunden ohne Ladesäulen-Anbindung: []
18:23:29 - __main__ - DEBUG - Routing-Knoten: 11 | Routing-Kanten: 110
18:23:30 - __main__ - DEBUG - Identische Fahrzeuggruppe: ['MAN_eTGX_1', 'MAN_eTGX_2', 'MAN_eTGX_3', 'MAN_eTGX_4']
18:23:30 - __main__ - DEBUG - Identische Fahrzeuggruppe: ['Volvo_FH_Electric_2', 'Volvo_FH_Electric_3']
18:23:30 - __main__ - DEBUG - Identische Fahrzeuggruppe: ['Mercedes_eActros_600_1', 'Mercedes_eActros_600_2']
18:23:30 - __main__ - DEBUG - ----------------------
18:23:30 - __main__ - DEBUG - Modellgröße
18:23:30 - __main__ - DEBUG - Anzahl x-Variablen: 990
18:23:30 - __main__ - DEBUG - Anzahl y-Variablen: 261
18:23:30 - __main__ - DEBUG - Anzahl p

c:\Users\j.beckmann\AppData\Local\anaconda3\Lib\site-packages\pulp\apis\coin_api.py:112: UserWarning: When using CBC on Windows, warmStart requires keepFiles=True.
  warnings.warn(


18:23:31 - pulp.apis.core - DEBUG - c:\Users\j.beckmann\AppData\Local\anaconda3\Lib\site-packages\pulp\apis\../solverdir/cbc/win/i64/cbc.exe C:\Users\J3898~1.BEC\AppData\Local\Temp\b40604fbbbd74120a026ffd357e91303-pulp.mps -mips C:\Users\J3898~1.BEC\AppData\Local\Temp\b40604fbbbd74120a026ffd357e91303-pulp.mst -sec 600 -ratio 0.02 -timeMode elapsed -branch -printingOptions all -solution C:\Users\J3898~1.BEC\AppData\Local\Temp\b40604fbbbd74120a026ffd357e91303-pulp.sol 


c:\Users\j.beckmann\AppData\Local\anaconda3\Lib\site-packages\pulp\apis\coin_api.py:203: UserWarning: `logPath` argument replaces `msg=1`. The output will be redirected to the log file.
  warnings.warn(



Status: Optimal
Objective (normiert): 0.07150761122819212
CBC: Result - Optimal solution found (within gap tolerance)
CBC: Objective value:                0.07150761
CBC: Lower bound:                    0.070
CBC: Gap:                            0.02

ROUTEN:
MAN_eTGX_1: DEPOT -> C3 -> C5 -> C6 -> C8 -> C10 -> DEPOT
	Kunden: 5 | Menge: 31000 l | reine Fahrzeit: 251 min | Arbeitszeit: 476 min (Service: 225, Laden: 0, Pause: 0)
MAN_eTGX_2: DEPOT -> C9 -> C2 -> C7 -> C1 -> C4 -> DEPOT
	Kunden: 5 | Menge: 26000 l | reine Fahrzeit: 235 min | Arbeitszeit: 460 min (Service: 225, Laden: 0, Pause: 0)
MAN_eTGX_3: nicht genutzt
MAN_eTGX_4: nicht genutzt
Volvo_FH_Electric_1: nicht genutzt
Volvo_FH_Electric_2: nicht genutzt
Volvo_FH_Electric_3: nicht genutzt
Mercedes_eActros_600_1: nicht genutzt
Mercedes_eActros_600_2: nicht genutzt

Zeit/Lenkzeit/Batterie je Fahrzeug :
  MAN_eTGX_1:
	C3: Zeit bei Ankunft=39 min | Verstrichene Zeit bis Pause=39/270 min | Reichweite=217 km | Restkapazität=26500 l
	

In [4]:
# ============================================================
# 13. JSON EXPORT (IM FORMAT DER HEURISTIK)
# ============================================================
import json
from datetime import datetime

def safe_val(var):
    """Hilfsfunktion: Verhindert Abstürze, falls der Solver None zurückgibt"""
    v = pl.value(var)
    return float(v) if v is not None else 0.0

# 1. Variablen initialisieren
solver_routes = {}
solver_trips = {}
solver_charging_side_trips = {}
solver_technical_routes = {}
served_customers_set = set()

makespan = 0.0
tot_dist = 0.0
tot_travel_min = 0.0
tot_service = 0.0
tot_charge_min = 0.0
tot_break_min = 0.0

if model.status == 1:  # Wenn eine gültige Lösung gefunden wurde
    # 2. Distanzen und Zeiten exakt berechnen
    for v in vehicles:
        for (i, j) in routing_arcs:
            if safe_val(x[(i, j, v)]) > 0.5:
                tot_dist += dist[(i, j)]
                tot_travel_min += time[(i, j)]
                if j != DEPOT:
                    tot_service += ServiceTime[j]

        for c in customers:
            if safe_val(p[(c, v)]) > 0.5:
                tot_break_min += BREAK_TIME
                
            for (cc, s) in side_pairs:
                if c == cc and safe_val(y[(c, s, v)]) > 0.5:
                    tot_dist += c_dist[(c, s)] + c_dist[(s, c)]
                    tot_travel_min += c_time[(c, s)] + c_time[(s, c)]
                    tot_charge_min += RECHARGE_TIME

    # 3. Routen und Lade-Abstecher je Fahrzeug extrahieren
    for v in vehicles:
        r = build_route(v)
        if len(r) > 2:  # Fahrzeug wurde aktiv genutzt
            solver_routes[v] = r
            solver_trips[v] = [r]
            solver_technical_routes[v] = [r]
            
            # Kunden als 'besucht' markieren
            for node in r:
                if node != DEPOT:
                    served_customers_set.add(node)
                    
            # Ladesäulen extrahieren
            charges = []
            last_cust = r[-2] # Der letzte Kunde vor der Rückkehr zum Depot
            
            for c in r:
                if c != DEPOT:
                    for (cc, s) in side_pairs:
                        if c == cc and safe_val(y[(c, s, v)]) > 0.5:
                            charges.append(s)
            solver_charging_side_trips[v] = charges
            
            # Makespan berechnen (Wann ist der Lkw zurück am Depot?)
            arrival_at_last_cust = safe_val(T[(last_cust, v)])
            side_t_val = sum(
                (c_time[(last_cust, s)] + c_time[(s, last_cust)] + RECHARGE_TIME)
                for (c, s) in side_pairs 
                if c == last_cust and safe_val(y[(c, s, v)]) > 0.5
            )
            break_t_val = BREAK_TIME if safe_val(p[(last_cust, v)]) > 0.5 else 0.0
            drive_back = time[(last_cust, DEPOT)]
            
            arrival_depot = arrival_at_last_cust + ServiceTime[last_cust] + side_t_val + break_t_val + drive_back
            if arrival_depot > makespan:
                makespan = arrival_depot
        else:
            # Fahrzeug blieb im Depot
            solver_routes[v] = []
            solver_trips[v] = []
            solver_technical_routes[v] = []
            solver_charging_side_trips[v] = []

# 4. Kundennamen extrahieren (falls in Rohdaten vorhanden)
customer_names_dict = {}
if "destination_name" in df_kunden_raw.columns:
    customer_names_dict = df_kunden_raw.set_index("id")["destination_name"].to_dict()

# 5. JSON-Struktur exakt nach Heuristik-Vorlage aufbauen
output_json = {
    "schema_version": "1.0",
    "status": "feasible" if model.status == 1 else "infeasible",
    "search_status": "optimal" if pl.LpStatus[model.status] == "Optimal" else "stopped_on_time_limit",
    "routes": solver_routes,
    "trips": solver_trips,
    "charging_side_trips": solver_charging_side_trips,
    "technical_routes": solver_technical_routes,
    "metadata": {
        "dataset_name": "Solver_Instance",
        "construction_strategy": "mip_solver",
        "included_customers": customers,
        "excluded_customers": [],
        "served_customers": list(served_customers_set),
        "unserved_customers": list(set(customers) - served_customers_set),
        "customer_names": customer_names_dict,
        "vehicle_ids": vehicles,
        "vehicle_hazard_compatibility": {v: ["3"] for v in vehicles},
        "vehicle_hazard_compatibility_source": "mip_solver_assumption",
        "risk_source": "solver_eval_cost_preprocessing",
        "single_trip_per_vehicle": True,
        "warnings": [
            "This JSON was generated dynamically from the PuLP MIP Solver."
        ]
    },
    "objective": {
        "value": safe_val(model.objective) if model.status == 1 else None,
        "weights": {
            "risk": W_RISK,
            "cost": W_COST,
            "time": W_TIME
        },
        "scales": {
            "risk": RISK_SCALE,
            "cost": COST_SCALE,
            "time": TIME_SCALE,
            "risk_active": True
        }
    },
    "metrics": {
        "total_risk": safe_val(risk_expr) if model.status == 1 else 0.0,
        "total_cost": safe_val(cost_expr) if model.status == 1 else 0.0,
        "total_activation_cost": safe_val(fixed_cost) if model.status == 1 else 0.0,
        "total_trip_cost": 0.0,
        "total_road_operating_cost": safe_val(travel_cost) if model.status == 1 else 0.0,
        "total_station_charging_cost": safe_val(side_trip_cost) if model.status == 1 else 0.0,
        "total_end_of_day_recharge_cost": 0.0,
        "total_distance_km": tot_dist,
        "total_travel_minutes": tot_travel_min,
        "total_service_minutes": tot_service,
        "total_waiting_minutes": 0.0,
        "total_charging_minutes": tot_charge_min,
        "total_break_minutes": tot_break_min,
        "total_time_minutes": safe_val(time_expr) if model.status == 1 else 0.0,
        "makespan_minute": makespan
    },
    "runtime_seconds": {
        "data_loading": t1 - t0,
        "data_preprocessing": t2 - t1,
        "model_building": t3 - t2,
        "model_solving": t4 - t3,
        "result_processing": t5 - t4,
        "total": t5 - t0
        
    },
    "infeasibility_reasons": [] if model.status == 1 else ["Solver could not find a feasible solution."]
}

# 6. JSON als Datei speichern
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
export_filename = f"solver_small_output_{TIMESTAMP}_R_{W_RISK}_C_{W_COST}_T_{W_TIME}.json"
try:
    with open(export_filename, "w", encoding="utf-8") as f:
        json.dump(output_json, f, indent=2, ensure_ascii=False)
    print(f"\nErfolg: Solver-Lösung im Heuristik-Format als '{export_filename}' gespeichert.")
except Exception as e:
    print(f"\nFehler beim Speichern der JSON: {e}")


Erfolg: Solver-Lösung im Heuristik-Format als 'solver_small_output_20260707_182627_R_0.5_C_0.3_T_0.2.json' gespeichert.
